In [0]:
dbutils.library.restartPython()

In [0]:
from parser.xmlreader import XMLReader
from parser.repository_parser import RepositoryParser
import json

XML_PATH = "/Volumes/idmc_poc/bronze/idmc_files/wf_s_m_TIM_STG_UNQ_WRKSTN_NOMTCH_SYNC_SANITIZE.XML"

# 1. Read XML
xml_root = XMLReader(XML_PATH).read()

print("ROOT TAG:", xml_root.tag)

# 2. Parse repository
parser = RepositoryParser()
repository = parser.parse(xml_root)

# 3. Validate repository format
print("\nFORMAT:", repository.get("format"))
print("REPOSITORY:", repository.get("name"))

# 4. Validate folders
folders = repository.get("folders", [])
print("\nFOLDER COUNT:", len(folders))

for folder in folders:
    print("\nFOLDER:", folder.get("name"))
    print("SESSIONS:", len(folder.get("sessions", [])))
    print("MAPPING REFERENCES:", len(folder.get("mapping_references", [])))

# 5. Validate session-to-mapping references
print("\nSESSION MAPPING REFERENCES:")

for ref in repository.get("mapping_references", []):
    print(
        "Session:", ref.get("session_name"),
        "| Mapping:", ref.get("mapping_name"),
        "| Scope:", ref.get("scope")
    )

# 6. Validate summary
print("\nSUMMARY:")
print(json.dumps(repository.get("summary", {}), indent=4))

# 7. Save parsed output for inspection
OUTPUT_PATH = "/Volumes/idmc_poc/bronze/idmc_files/parsed_metadata/powercenter_parser_test.json"

dbutils.fs.put(
    OUTPUT_PATH,
    json.dumps(repository, indent=4, default=str),
    overwrite=True
)

print("\nPARSER TEST COMPLETED")
print("OUTPUT:", OUTPUT_PATH)

In [0]:
from parser.xmlreader import XMLReader
from parser.session_parser import SessionParser

XML_PATH = "/Volumes/idmc_poc/bronze/idmc_files/wf_s_m_TIM_STG_UNQ_WRKSTN_NOMTCH_SYNC_SANITIZE.XML"

root = XMLReader(XML_PATH).read()

folder = next(
    child for child in root
    if child.tag.upper() == "REPOSITORY"
).find("FOLDER")

parser = SessionParser()
sessions = parser.parse(folder)

print("SESSION COUNT:", len(sessions))

for session in sessions:
    print("\nSESSION:", session["name"])
    print("MAPPING:", session["mapping_name"])
    print("TRANSFORMATIONS:", len(session["transformation_instances"]))
    print("EXTENSIONS:", len(session["session_extensions"]))
    print("CONNECTION REFERENCES:", len(session["connection_references"]))
    print("SESSION ATTRIBUTES:", len(session["session_attributes"]))

assert len(sessions) == 4
assert sum(len(s["transformation_instances"]) for s in sessions) == 32
assert sum(len(s["session_extensions"]) for s in sessions) == 22
assert sum(len(s["connection_references"]) for s in sessions) == 16

print("\nSESSION PARSER TEST PASSED")

In [0]:
import importlib
import parser.session_parser as session_module
from parser.xmlreader import XMLReader

# Reload the module to avoid stale imports
session_module = importlib.reload(session_module)
SessionParser = session_module.SessionParser

root = XMLReader(XML_PATH).read()

print("ROOT:", root.tag)

repository = next(
    (child for child in root if child.tag.upper() == "REPOSITORY"),
    None
)

print("REPOSITORY FOUND:", repository is not None)

folder = next(
    (child for child in repository if child.tag.upper() == "FOLDER"),
    None
) if repository is not None else None

print("FOLDER FOUND:", folder is not None)
print("FOLDER TAG:", folder.tag if folder is not None else None)

if folder is not None:
    print("DIRECT CHILD TAGS:", [child.tag for child in folder])
    print(
        "DIRECT SESSION COUNT:",
        sum(child.tag.upper() == "SESSION" for child in folder)
    )

print("IMPORTED MODULE:", session_module.__file__)

parser = SessionParser()
sessions = parser.parse(folder)

print("PARSER SESSION COUNT:", len(sessions))

for session in sessions:
    print(session.get("name"), "->", session.get("mapping_name"))

In [0]:
import xml.etree.ElementTree as ET
from collections import Counter

XML_PATH = "/Volumes/idmc_poc/bronze/idmc_files/wf_s_m_TIM_STG_UNQ_WRKSTN_NOMTCH_SYNC_SANITIZE.XML"

tree = ET.parse(XML_PATH)
root = tree.getroot()

print("ROOT:", root.tag)
print("ROOT CHILDREN:", [child.tag for child in root])

repository = next(
    (child for child in root if child.tag.upper() == "REPOSITORY"),
    None
)

print("REPOSITORY FOUND:", repository is not None)

folder = next(
    (child for child in repository if child.tag.upper() == "FOLDER"),
    None
)

print("FOLDER:", folder.get("NAME"))
print("FOLDER CHILD COUNTS:", Counter(child.tag.upper() for child in folder))

sessions = [
    child for child in folder
    if child.tag.upper() == "SESSION"
]

print("DIRECT SESSIONS:", len(sessions))

for session in sessions[:10]:
    print(
        "SESSION:",
        session.get("NAME"),
        "| MAPPING:",
        session.get("MAPPINGNAME")
    )

In [0]:
dbutils.library.restartPython()

In [0]:
from parser.xmlreader import XMLReader
from parser.mapping_parser import MappingParser

XML_PATH = "/Volumes/idmc_poc/bronze/idmc_files/wf_s_m_TIM_STG_UNQ_WRKSTN_NOMTCH_SYNC_SANITIZE.XML"

root = XMLReader(XML_PATH).read()

repository = next(
    child for child in root
    if child.tag.upper() == "REPOSITORY"
)

folder = next(
    child for child in repository
    if child.tag.upper() == "FOLDER"
)

parser = MappingParser()
mappings = parser.parse(folder)

print("MAPPING DEFINITION COUNT:", len(mappings))

for mapping in mappings:
    print("\nMAPPING:", mapping["name"])
    print("SOURCES:", len(mapping["source_instances"]))
    print("TRANSFORMATIONS:", len(mapping["transformations"]))
    print("TARGETS:", len(mapping["target_instances"]))
    print("CONNECTORS:", len(mapping["links"]))

assert len(mappings) == 0

print("\nMAPPING PARSER TEST PASSED")
print("No mapping definitions were fabricated.")

In [0]:
from parser.source_parser import SourceParser
from parser.target_parser import TargetParser
from parser.connector_parser import ConnectorParser
from parser.port_parser import PortParser
from parser.xmlreader import XMLReader

XML_PATH = "/Volumes/idmc_poc/bronze/idmc_files/wf_s_m_TIM_STG_UNQ_WRKSTN_NOMTCH_SYNC_SANITIZE.XML"

root = XMLReader(XML_PATH).read()

repository = next(
    child for child in root
    if child.tag.upper() == "REPOSITORY"
)

folder = next(
    child for child in repository
    if child.tag.upper() == "FOLDER"
)

source_parser = SourceParser()
target_parser = TargetParser()
connector_parser = ConnectorParser()
port_parser = PortParser()

sources = source_parser.parse(folder)
targets = target_parser.parse(folder)
connections = connector_parser.parse(folder)
connection_refs = connector_parser.parse_references(folder)

print("SOURCES:", len(sources))
print("TARGETS:", len(targets))
print("CONNECTION DEFINITIONS:", len(connections))
print("CONNECTION REFERENCES:", len(connection_refs))

assert len(sources) == 0
assert len(targets) == 0
assert len(connections) == 0
assert len(connection_refs) == 25

print("\nALL FOUR PARSER TESTS PASSED")

In [0]:
from parser.graph_builder import GraphBuilder

builder = GraphBuilder()

# Test actual current XML mapping result
mapping = {
    "name": "m_TIM_DML_ATLAS_SRC_TGT_CNT_gen",
    "source_instances": [],
    "transformations": [],
    "target_instances": [],
    "links": []
}

graph = builder.build(mapping)

print("STATUS:", graph["status"])
print("NODES:", len(graph["nodes"]))
print("EDGES:", len(graph["edges"]))
print("EXECUTION ORDER:", graph["execution_order"])

assert graph["status"] == "NO_MAPPING_DEFINITIONS"
assert len(graph["nodes"]) == 0
assert len(graph["edges"]) == 0

print("\nGRAPH BUILDER TEST PASSED")

In [0]:

import os
import json
import xml.etree.ElementTree as ET

from parser.repository_parser import RepositoryParser


# ==========================================================
# 1. CONFIGURATION
# ==========================================================

XML_PATH = (
    "/Volumes/idmc_poc/bronze/idmc_files/wf_s_m_TIM_STG_UNQ_WRKSTN_NOMTCH_SYNC_SANITIZE.XML"
)

OUTPUT_PATH = (
    "/Volumes/idmc_poc/bronze/idmc_files/"
    "parsed_metadata/repository_integration_test.json"
)


# ==========================================================
# 2. CHECK XML FILE
# ==========================================================

print("=" * 70)
print("REPOSITORY PARSER - INTEGRATION TEST")
print("=" * 70)

assert os.path.exists(XML_PATH), (
    f"XML file not found: {XML_PATH}"
)

print("XML FILE:", XML_PATH)


# ==========================================================
# 3. LOAD XML
# ==========================================================

tree = ET.parse(XML_PATH)
root = tree.getroot()

print("XML ROOT:", root.tag)


# ==========================================================
# 4. INITIALIZE REPOSITORY PARSER
# ==========================================================

repository_parser = RepositoryParser()

repository = repository_parser.parse(root)

assert repository is not None, "Repository parsing returned None"

assert repository.get("format") == "POWERCENTER", (
    f"Unexpected format: {repository.get('format')}"
)


# ==========================================================
# 5. REPOSITORY VALIDATION
# ==========================================================

print("\n[1] REPOSITORY")

print("Name:", repository.get("name"))
print("Format:", repository.get("format"))

folders = repository.get("folders", [])

assert len(folders) == 1, (
    f"Expected 1 folder, found {len(folders)}"
)

folder = folders[0]

print("\n[2] FOLDER")
print("Folder:", folder.get("name"))


# ==========================================================
# 6. CONNECTED PARSER VALIDATION
# ==========================================================

print("\n[3] PARSER INTEGRATION")

sources = folder.get("sources", [])
targets = folder.get("targets", [])
connections = folder.get("connections", [])
mappings = folder.get("mappings", [])
sessions = folder.get("sessions", [])

print("Sources:", len(sources))
print("Targets:", len(targets))
print("Connections:", len(connections))
print("Mapping definitions:", len(mappings))
print("Sessions:", len(sessions))


# ==========================================================
# 7. SESSION VALIDATION
# ==========================================================

print("\n[4] SESSION DETAILS")

for session in sessions:
    print(
        session.get("name"),
        "->",
        session.get("mapping_name")
    )

assert len(sessions) == 4, (
    f"Expected 4 sessions, found {len(sessions)}"
)


# ==========================================================
# 8. MAPPING REFERENCE VALIDATION
# ==========================================================

print("\n[5] MAPPING REFERENCES")

mapping_references = repository.get(
    "mapping_references", []
)

for mapping in mapping_references:
    print(
        mapping.get("session_name"),
        "->",
        mapping.get("mapping_name")
    )

assert len(mapping_references) == 4, (
    f"Expected 4 mapping references, "
    f"found {len(mapping_references)}"
)


# ==========================================================
# 9. WORKFLOW VALIDATION
# ==========================================================

print("\n[6] WORKFLOW DETAILS")

workflows = folder.get("workflows", [])

for workflow in workflows:
    print("Workflow:", workflow.get("name"))
    print("Tasks:", len(workflow.get("tasks", [])))
    print(
        "Workflow variables:",
        len(workflow.get("workflow_variables", []))
    )

assert len(workflows) == 1, (
    f"Expected 1 workflow, found {len(workflows)}"
)

standalone_tasks = folder.get("tasks", [])
configurations = folder.get("configurations", [])

print("Standalone task instances:", len(standalone_tasks))
print("Configurations:", len(configurations))


# ==========================================================
# 10. REPOSITORY SUMMARY
# ==========================================================

print("\n[7] REPOSITORY SUMMARY")

summary = repository.get("summary", {})

for key, value in summary.items():
    print(f"{key}: {value}")


# ==========================================================
# 11. INTEGRATION ASSERTIONS
# ==========================================================

print("\n[8] VALIDATING EXPECTED RESULTS")

expected_counts = {
    "folder_count": 1,
    "session_count": 4,
    "workflow_count": 1,
    "workflow_task_instance_count": 15,
    "workflow_variable_count": 160,
    "session_transformation_instance_count": 32,
    "session_extension_count": 22,
    "connection_reference_count": 16,
    "mapping_reference_count": 4
}

for key, expected in expected_counts.items():
    actual = summary.get(key)

    assert actual == expected, (
        f"{key}: expected {expected}, found {actual}"
    )

    print(f"PASS: {key} = {actual}")


# Actual extracted folder-level results from your test
assert len(sources) == 6, (
    f"Expected 6 source instances, found {len(sources)}"
)

assert len(targets) == 7, (
    f"Expected 7 target instances, found {len(targets)}"
)

assert len(connections) == 8, (
    f"Expected 8 connection references, found {len(connections)}"
)

assert len(mappings) == 0, (
    "Unexpected mapping definitions found. "
    "Review the XML before proceeding."
)

assert summary.get("mapping_definition_count") == 0
assert summary.get("mapping_definitions_present") is False

print("PASS: Source instances = 6")
print("PASS: Target instances = 7")
print("PASS: Connection references = 8")
print("PASS: Mapping definitions = 0 (expected)")


# ==========================================================
# 12. VERIFY SESSION TRANSFORMATION INSTANCE COUNTS
# ==========================================================

print("\n[9] SESSION TRANSFORMATION DETAILS")

total_instances = 0

for session in sessions:
    instance_count = len(
        session.get("transformation_instances", [])
    )

    total_instances += instance_count

    print(
        session.get("name"),
        "->",
        instance_count,
        "transformation instances"
    )

assert total_instances == 32, (
    f"Expected 32 transformation instances, "
    f"found {total_instances}"
)


# ==========================================================
# 13. SAVE PARSED REPOSITORY JSON
# ==========================================================

print("\n[10] SAVING PARSED METADATA")

output_dir = os.path.dirname(OUTPUT_PATH)

os.makedirs(output_dir, exist_ok=True)

with open(OUTPUT_PATH, "w", encoding="utf-8") as file:
    json.dump(
        repository,
        file,
        indent=4,
        ensure_ascii=False,
        default=str
    )

assert os.path.exists(OUTPUT_PATH), (
    "Output JSON was not created"
)

print("JSON SAVED:", OUTPUT_PATH)
print("FILE SIZE:", os.path.getsize(OUTPUT_PATH), "bytes")


# ==========================================================
# 14. FINAL RESULT
# ==========================================================

print("\n" + "=" * 70)
print("ALL REPOSITORY PARSER INTEGRATION TESTS PASSED")
print("=" * 70)

print("\nFINAL COUNTS")
print("Folders:", len(folders))
print("Sessions:", len(sessions))
print("Sources:", len(sources))
print("Targets:", len(targets))
print("Connections:", len(connections))
print("Mapping definitions:", len(mappings))
print("Mapping references:", len(mapping_references))
print("Workflows:", len(workflows))
print("Workflow task instances:", summary["workflow_task_instance_count"])
print("Workflow variables:", summary["workflow_variable_count"])
print("Transformation instances:", total_instances)

print("\nNEXT STEP: MIGRATION PIPELINE INTEGRATION")


In [0]:
import os

for root, dirs, files in os.walk("/Workspace/Users"):
    if "migration_pipeline.py" in files:
        print("FOUND:", os.path.join(root, "migration_pipeline.py"))

In [0]:
import os

PIPELINE_PATH = "/Workspace/Users/lokeswaraseetaramasastry.k@tcs.com/IDMC_Databricks_LLM/pipeline/migration_pipeline.py"

with open(PIPELINE_PATH, "r", encoding="utf-8") as f:
    source_code = f.read()

compile(source_code, PIPELINE_PATH, "exec")

print("PASS: migration_pipeline.py syntax is valid")
print("File size:", len(source_code), "characters")

In [0]:
%pip install openai

In [0]:
import sys
import importlib

PROJECT_PATH = "/Workspace/Users/lokeswaraseetaramasastry.k@tcs.com/IDMC_Databricks_LLM"

if PROJECT_PATH not in sys.path:
    sys.path.insert(0, PROJECT_PATH)

import pipeline.migration_pipeline as migration_module

importlib.reload(migration_module)

MigrationPipeline = migration_module.MigrationPipeline

print("PASS: MigrationPipeline imported successfully")
print("CLASS:", MigrationPipeline)

In [0]:
pipeline = MigrationPipeline(
    xml_path=XML_PATH,
    spark=spark,
    workspace_url=WORKSPACE_URL,
    databricks_token=DATABRICKS_TOKEN,
    llm_model=LLM_MODEL,
    selected_mapping_name=None,
    dry_run=True,
    execution_approved=False,
    allow_synthetic_merge_keys=False,
    verbose=True
)

results = pipeline.run()

print("\n========== FINAL PIPELINE RESULTS ==========")
print(results)

In [0]:
# ==========================================
# POWER CENTER MIGRATION PIPELINE - DRY RUN
# ==========================================

import os
import sys
import importlib

PROJECT_PATH = (
    "/Workspace/Users/lokeswaraseetaramasastry.k@tcs.com/"
    "IDMC_Databricks_LLM"
)

XML_PATH = (
    "/Volumes/idmc_poc/bronze/idmc_files/"
    "wf_s_m_TIM_STG_UNQ_WRKSTN_NOMTCH_SYNC_SANITIZE.XML"
)

OUTPUT_DIR = (
    "/Volumes/idmc_poc/bronze/idmc_files/"
    "parsed_metadata"
)

# 1. Validate paths
assert os.path.exists(XML_PATH), f"XML not found: {XML_PATH}"
assert os.path.exists(PROJECT_PATH), f"Project not found: {PROJECT_PATH}"

os.makedirs(OUTPUT_DIR, exist_ok=True)

# 2. Load project
if PROJECT_PATH not in sys.path:
    sys.path.insert(0, PROJECT_PATH)

import pipeline.migration_pipeline as migration_module
importlib.reload(migration_module)

MigrationPipeline = migration_module.MigrationPipeline

# 3. Get Databricks workspace configuration
WORKSPACE_URL = spark.conf.get("spark.databricks.workspaceUrl")
WORKSPACE_URL = (
    WORKSPACE_URL
    if WORKSPACE_URL.startswith("https://")
    else f"https://{WORKSPACE_URL}"
)

DATABRICKS_TOKEN = (
    dbutils.notebook.entry_point
    .getDbutils()
    .notebook()
    .getContext()
    .apiToken()
    .get()
)

LLM_MODEL = "databricks-gpt-oss-120b"

# 4. Initialize safely
pipeline = MigrationPipeline(
    xml_path=XML_PATH,
    spark=spark,
    workspace_url=WORKSPACE_URL,
    databricks_token=DATABRICKS_TOKEN,
    llm_model=LLM_MODEL,
    selected_mapping_name=None,
    dry_run=True,
    execution_approved=False,
    allow_synthetic_merge_keys=False,
    verbose=True
)

# 5. Run
results = pipeline.run()

print("\n========== FINAL PIPELINE RESULTS ==========")
print(results)

----------------------------------------------------------

In [0]:
#1
# ============================================================
# DISPLAY GENERATED PYSPARK CODE FOR ALL MAPPINGS
# ============================================================

GENERATED_DIR = (
    "/Volumes/idmc_poc/bronze/idmc_files/generated_pyspark"
)

MAPPINGS = [
    "m_customer_standardize",
    "m_order_enrichment",
    "m_inventory_reconciliation",
    "m_daily_sales_summary"
]

for mapping_name in MAPPINGS:

    file_path = f"{GENERATED_DIR}/{mapping_name}.py"

    print("\n" + "=" * 100)
    print(f"GENERATED PYSPARK: {mapping_name}")
    print("=" * 100)

    try:
        code = dbutils.fs.head(file_path, 1000000)

        print(code)

        print("\nFile:", file_path)
        print("Status: Successfully read")

    except Exception as e:
        print("Unable to read generated code:", str(e))


In [0]:
parsed_path = (
    "/Volumes/idmc_poc/bronze/idmc_files/"
    "parsed_metadata/idmc_parsed_objects.json"
)

parsed_df = spark.read.option("multiline", "true").json(parsed_path)

display(parsed_df)

In [0]:
# ============================================================
# FINAL POST-MIGRATION VALIDATION - ALL FOUR TARGETS
# ============================================================

from pyspark.sql import functions as F

TARGETS = {
    "m_customer_standardize": {
        "table": "idmc_poc.migration_test.mig4_customer_standardized",
        "keys": ["customer_id"]
    },
    "m_order_enrichment": {
        "table": "idmc_poc.migration_test.mig4_order_enriched",
        "keys": ["order_id"]
    },
    "m_inventory_reconciliation": {
        "table": "idmc_poc.migration_test.mig4_inventory_position",
        "keys": ["product_id", "warehouse_id"]
    },
    "m_daily_sales_summary": {
        "table": "idmc_poc.migration_test.mig4_daily_sales_summary",
        "keys": ["order_date"]
    }
}

final_results = []

for mapping, config in TARGETS.items():

    table = config["table"]
    keys = config["keys"]

    df = spark.table(table)

    null_condition = F.lit(False)

    for key in keys:
        null_condition = null_condition | F.col(key).isNull()

    null_rows = df.filter(null_condition).count()

    duplicate_groups = (
        df.groupBy(*keys)
          .count()
          .filter(F.col("count") > 1)
          .count()
    )

    result = {
        "mapping": mapping,
        "target": table,
        "row_count": df.count(),
        "column_count": len(df.columns),
        "null_key_rows": null_rows,
        "duplicate_key_groups": duplicate_groups,
        "status": (
            "PASS"
            if null_rows == 0 and duplicate_groups == 0
            else "REVIEW_REQUIRED"
        )
    }

    final_results.append(result)

    print(result)

print("\nFINAL VALIDATION:",
      "ALL PASS" if all(
          x["status"] == "PASS" for x in final_results
      ) else "REVIEW REQUIRED")

In [0]:
from pathlib import Path

OUTPUT_DIR = Path(
    "/Volumes/idmc_poc/bronze/idmc_files/generated_pyspark"
)

mapping_files = [
    "m_customer_standardize.py",
    "m_order_enrichment.py",
    "m_inventory_reconciliation.py",
    "m_daily_sales_summary.py"
]

for filename in mapping_files:
    file_path = OUTPUT_DIR / filename

    print("\n" + "=" * 100)
    print(f"MAPPING FILE: {filename}")
    print(f"LOCATION: {file_path}")
    print("=" * 100)

    if file_path.exists():
        print(file_path.read_text(encoding="utf-8"))
    else:
        print("File not found:", file_path)